# 01 · Curva de descuento de Seguros Barú

Este cuaderno construye las curvas con las que Seguros Barú descuenta sus flujos de cumplimiento en los cierres de 2023, 2024 y 2025. IFRS 17 pide que las tasas reflejen el valor temporal del dinero, las características de los flujos y su liquidez (par. 36). También permite construirlas de abajo hacia arriba: una curva líquida libre de riesgo más una prima de iliquidez (B80).

El método tiene cinco pasos:

1. **Curva base:** la curva TNC del Tesoro de EE. UU., con tasas spot de fin de mes de 0,5 a 100 años.
2. **Diferencial:** la curva HQM de bonos corporativos AAA–A menos la TNC, plazo por plazo.
3. **Prima de iliquidez:** la política elegida suma el diferencial completo, así que la curva es la HQM. Como opciones quedan deducir la parte de crédito y graduar por liquidez.
4. **Curva mensual:** TNC más la prima, en tasa anual efectiva, interpolada a plazos mensuales.
5. **Control cruzado:** la TNC se compara con una curva Smith-Wilson construida con las tasas par diarias del Tesoro.

In [1]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
from ifrs17.treasury_curves import read_treasury_spot_file, build_discount_curve, semiannual_to_annual
from ifrs17.illiquidity import (CREDIT_SHARE, CREDIT_SHARE_CDS, APPLICATION_RATIO, APPLICATION_RATIO_BY_LIQUIDITY,
                                PORTFOLIO_LIQUIDITY, credit_share_from_cds, expected_loss_floor)
from ifrs17.curves import UFR_BY_YEAR, curve_from_par, load_par_curves
pd.set_option('display.precision', 3)
cierres = [pd.Period(m, 'M') for m in ('2023-12', '2024-12', '2025-12')]

## 1. Las curvas TNC y HQM del Tesoro

El Tesoro publica dos curvas con la misma metodología y la misma malla, cada medio año de 0,5 a 100 años, con capitalización semestral:

- **TNC** (*Treasury Nominal Coupon-issue*): se ajusta a los bonos del Tesoro, que tienen un riesgo de crédito nulo o insignificante (B79).
- **HQM** (*High Quality Market*): se ajusta a bonos corporativos con calificación AAA, AA y A.

Más allá de 30 años, el Tesoro mantiene constante la tasa forward en su promedio de 15 a 30 años, así que no hace falta otra extrapolación. Se usan las versiones de fin de mes, que reflejan las condiciones a la fecha de medición (par. 33(c), B54).

In [2]:
tnc = read_treasury_spot_file('../data/raw/treasury/tnceom_23_27.xlsx')
hqm = pd.concat([read_treasury_spot_file(f'../data/raw/treasury/{f}') for f in ('hqmeom_19_23.xls', 'hqmeom_24_28.xls')])
hqm = hqm[~hqm.index.duplicated(keep='last')].sort_index()
plazos = [0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0, 50.0, 100.0]
tabla = pd.concat({'TNC': 100 * tnc.loc[cierres, plazos], 'HQM': 100 * hqm.loc[cierres, plazos]})
tabla.round(2)

maturity_years  0.5    1.0    2.0    5.0    10.0   20.0   30.0   50.0   100.0
    month                                                                    
TNC 2023-12      5.17   4.78   4.21   3.88   3.75   4.30   4.04   4.22   4.36
    2024-12      4.25   4.24   4.25   4.42   4.58   4.99   4.86   4.98   5.07
    2025-12      3.56   3.52   3.49   3.74   4.22   5.03   5.04   5.28   5.46
HQM 2023-12      5.29   5.12   4.84   4.58   4.91   5.11   5.10   5.11   5.11
    2024-12      4.56   4.59   4.65   4.90   5.36   5.76   5.79   5.88   5.95
    2025-12      3.80   3.84   3.91   4.25   4.96   5.76   6.04   6.28   6.46

## 2. El diferencial HQM − TNC y la decisión sobre la parte de crédito

La diferencia entre las dos curvas mezcla tres cosas: la pérdida esperada por incumplimiento, una prima por el riesgo de crédito y una prima por la menor liquidez de los bonos corporativos.

**Política elegida:** sumar el diferencial completo a la TNC. Como TNC + (HQM − TNC) = HQM, la curva de descuento es la HQM. Es un juicio contable: se reconoce que el diferencial incluye compensación por riesgo de crédito, que B79 pide excluir, y por eso se revela la sensibilidad a deducirla (par. 117(c)(iii), 128).

**Opción disponible:** deducir la parte de crédito con un porcentaje. El motor acepta cualquier valor, y el apartado siguiente explica cómo calcularlo.

In [3]:
dif = pd.DataFrame({str(m): 1e4 * (hqm.loc[m] - tnc.loc[m]) for m in cierres}).T
print(f'Política: {CREDIT_SHARE:.0%} atribuido a crédito. Opción: {CREDIT_SHARE_CDS:.0%}.')
pd.concat({'diferencial HQM − TNC = prima de la política (pb)': dif[[1.0, 5.0, 10.0, 20.0, 30.0]],
           f'prima con la opción de {CREDIT_SHARE_CDS:.0%} (pb)': dif[[1.0, 5.0, 10.0, 20.0, 30.0]] * (1 - CREDIT_SHARE_CDS)}, axis=1).round(1)

Política: 0% atribuido a crédito. Opción: 55%.


diferencial HQM − TNC = prima de la política (pb)               \
maturity_years                                              1.0   5.0    10.0   
2023-12                                                     34.0  70.0  116.0   
2024-12                                                     35.0  48.0   78.0   
2025-12                                                     32.0  51.0   74.0   

                            prima con la opción de 55% (pb)                    \
maturity_years  20.0   30.0                            1.0   5.0   10.0  20.0   
2023-12         81.0  106.0                            15.3  31.5  52.2  36.5   
2024-12         77.0   93.0                            15.7  21.6  35.1  34.6   
2025-12         73.0  100.0                            14.4  23.0  33.3  32.8   

                      
maturity_years  30.0  
2023-12         47.7  
2024-12         41.8  
2025-12         45.0

### Cómo se calcula el porcentaje de crédito (para revisiones futuras)

IFRS 17 no fija un porcentaje (B78). Pide maximizar datos observables (B78(a)), excluir el riesgo de crédito (B79) y sugiere los derivados de crédito como referencia (B85). El procedimiento documentado es:

1. **Participación del incumplimiento según CDS.** Para cada calificación, prima del CDS ÷ diferencial del bono sobre el Tesoro. Longstaff, Mithal y Neis (2005) midieron 51% en AAA/AA y 56% en A.
2. **Ponderación por calificación.** Se ponderan esas participaciones con la mezcla de calificaciones de los bonos de la HQM (AAA a A).
3. **Piso de pérdida esperada.** El porcentaje debe cubrir al menos la pérdida esperada de un bono A: probabilidad anual de incumplimiento × severidad ÷ diferencial. Se usa el mayor entre los pasos 2 y 3.
4. **Consistencia.** Se aplica el mismo porcentaje en todos los cierres salvo evidencia nueva (B54–B55), y se revisa al menos una vez al año.
5. **Revelación.** Se describen el método y las curvas, y la sensibilidad del pasivo al porcentaje (par. 117(c)(iii), 120, 128).

La mezcla de calificaciones y la probabilidad de incumplimiento de abajo son valores ilustrativos que hay que actualizar en cada revisión.

In [4]:
paso2 = credit_share_from_cds({'AAA/AA': 0.2, 'A': 0.8})
diferencial_10 = dif.loc['2025-12', 10.0] / 1e4
paso3 = expected_loss_floor(0.0008, 0.60, diferencial_10)
print(f'Paso 2 (CDS ponderado): {paso2:.1%}  |  Paso 3 (piso de pérdida esperada): {paso3:.1%}  |  Sugerido: {max(paso2, paso3):.1%}')

Paso 2 (CDS ponderado): 55.0%  |  Paso 3 (piso de pérdida esperada): 6.5%  |  Sugerido: 55.0%


## 3. Factor por liquidez del pasivo

La política aplica el mismo factor de 100% a todos los portafolios. La opción graduada ajusta la prima según cuán difícil es para el asegurado exigir el pago antes de tiempo: vida temporal no tiene valor de rescate, vida con ahorro sí. El reaseguro usa siempre la misma curva que su subyacente (par. 63).

In [5]:
pd.DataFrame({'nivel': PORTFOLIO_LIQUIDITY,
              'factor de la política': {p: APPLICATION_RATIO[b] for p, b in PORTFOLIO_LIQUIDITY.items()},
              'factor de la opción graduada': {p: APPLICATION_RATIO_BY_LIQUIDITY[b] for p, b in PORTFOLIO_LIQUIDITY.items()}})

,nivel,factor de la política,factor de la opción graduada
VT,high,1.0,1.00
RE_VT,high,1.0,1.00
AUTO,medium,1.0,0.75
INC,medium,1.0,0.75
TRC,medium,1.0,0.75
VA,low,1.0,0.50


## 4. Curva de descuento y tabla mensual

La curva se pasa de capitalización semestral a tasa anual efectiva y se interpola a plazos mensuales, de 1 a 1.200 meses. La interpolación es lineal sobre el logaritmo del factor de descuento, lo que equivale a una tasa forward constante dentro de cada semestre. Por debajo de 6 meses se mantiene la tasa de 0,5 años.

In [6]:
mat = [1, 5, 10, 20, 30, 40, 60, 100]
resumen = {}
for m in cierres:
    resumen[(str(m), 'política = HQM')] = 100 * build_discount_curve(tnc.loc[m], hqm.loc[m], CREDIT_SHARE, 1.0).spot(mat)
    resumen[(str(m), f'opción {CREDIT_SHARE_CDS:.0%} crédito')] = 100 * build_discount_curve(tnc.loc[m], hqm.loc[m], CREDIT_SHARE_CDS, 1.0).spot(mat)
    resumen[(str(m), 'solo TNC')] = 100 * build_discount_curve(tnc.loc[m], hqm.loc[m], 1.0, 1.0).spot(mat)
pd.DataFrame(resumen, index=pd.Index(mat, name='plazo (años)')).T.round(3)

plazo (años)                  1      5      10     20     30     40     60   \
2023-12 política = HQM      5.186  4.632  4.970  5.175  5.165  5.165  5.175   
        opción 55% crédito  4.994  4.239  4.318  4.719  4.568  4.630  4.702   
        solo TNC            4.837  3.918  3.785  4.346  4.081  4.193  4.316   
2024-12 política = HQM      4.643  4.960  5.432  5.843  5.874  5.936  5.997   
        opción 55% crédito  4.446  4.690  4.992  5.408  5.348  5.421  5.488   
        solo TNC            4.285  4.469  4.632  5.052  4.919  5.001  5.073   
2025-12 política = HQM      3.877  4.295  5.022  5.843  6.131  6.286  6.440   
        opción 55% crédito  3.698  4.009  4.605  5.430  5.565  5.720  5.874   
        solo TNC            3.551  3.775  4.265  5.093  5.104  5.257  5.411   

plazo (años)                  100  
2023-12 política = HQM      5.175  
        opción 55% crédito  4.753  
        solo TNC            4.408  
2024-12 política = HQM      6.039  
        opción 55% crédito  5.541  
        solo TNC            5.134  
2025-12 política = HQM      6.564  
        opción 55% crédito  5.997  
        solo TNC            5.535

In [7]:
vt_2025 = build_discount_curve(tnc.loc[cierres[2]], hqm.loc[cierres[2]], CREDIT_SHARE, 1.0)
with pd.option_context('display.precision', 6):
    display(vt_2025.monthly_table().loc[[1, 6, 12, 60, 120, 240, 360, 600, 1200]])

,discount_factor,spot_annual,spot_monthly
maturity_months,,,
1,0.996868,0.038361,0.003142
6,0.981354,0.038361,0.003142
12,0.962678,0.038769,0.003175
60,0.810362,0.042952,0.003511
120,0.612657,0.050215,0.004091
240,0.321190,0.058429,0.004743
360,0.167767,0.061312,0.004971
600,0.045424,0.063786,0.005166
1200,0.001733,0.065643,0.005312


![Curva IFRS 17 frente a TNC y HQM](../docs/figures/curvas_cierre.png)

Con la política elegida, la curva de IFRS 17 es la HQM, entre 72 y 123 pb por encima del Tesoro de 10 a 30 años. La opción con 55% de crédito queda en medio. Entre 2023 y 2025 la curva se empinó: el plazo de 1 año bajó y el de 20 años subió. Para vida temporal y vida con ahorro, que tienen flujos largos, eso reduce el valor presente de las obligaciones. Ese efecto es financiero, así que no toca el CSM (B97(a)) y va a ingresos o gastos financieros de seguros.

## 5. Control cruzado con Smith-Wilson

La TNC es un ajuste del propio Tesoro. Como comprobación independiente, se construye una curva Smith-Wilson con las tasas par diarias del último día hábil de cada cierre y se comparan hasta 30 años. Las diferencias de algunos puntos básicos vienen de que ambas curvas usan bonos y métodos de ajuste distintos.

In [8]:
par = load_par_curves('../data/raw/treasury/par_month_end_2023_2025.csv')
fechas = {'2023-12': '2023-12-29', '2024-12': '2024-12-31', '2025-12': '2025-12-31'}
filas = []
for m in cierres:
    sw = curve_from_par(par.loc[fechas[str(m)]], UFR_BY_YEAR[m.year])
    for t in (1, 2, 5, 10, 20, 30):
        a = semiannual_to_annual(tnc.loc[m, float(t)])
        filas.append({'cierre': str(m), 'plazo': t, 'TNC %': 100 * a, 'Smith-Wilson %': 100 * sw.spot(t)[0],
                      'diferencia pb': 1e4 * (a - sw.spot(t)[0])})
pd.DataFrame(filas).pivot(index='plazo', columns='cierre', values='diferencia pb').round(1)

cierre,2023-12,2024-12,2025-12
plazo,,,
1,-0.4,8.3,4.2
2,-0.5,-0.2,2.1
5,6.1,3.1,-0.3
10,-12.6,-3.4,-2.7
20,1.7,2.3,-0.7
30,6.9,8.6,1.1


## Qué queda pendiente

- Revisar el porcentaje de crédito y la mezcla de calificaciones al menos una vez al año, aunque la política actual no lo deduzca, para mantener actualizada la sensibilidad.
- Calcular las curvas bloqueadas de cada cohorte con las primas del simulador (B73); `data/processed/discount_curves_halfyear.csv` ya tiene la curva de cada fin de mes de 2023–2025 por nivel de liquidez.